# Construction Safety Vision — treino, avaliação e inferência

**Mode A (default):** clone → no project dependency installation → dataset, recorded training recipe and curves, committed evaluation results, validation FP/FN figures and real-video figures. No checkpoints, GPU, dataset credentials or dataset files required.

**Mode B (optional):** install the locked inference environment → upload exact frozen D2/S1 → verify SHA-256 → upload your external MP4 → execute the existing video runtime.

Reading order: **1** overview, **2** dataset and classes, **3** TREINO, **4** AVALIAÇÃO, **5** qualitative evidence, **6** video evidence, **7** INFERÊNCIA, **8** limitations and licences.

Sections **3.1-3.3**, **4.1-4.2** and **7.1** state the models, the training recipe, the metrics and the operating points in prose, so the methodology can be read without opening a JSON artifact or a source file.

The scientific lifecycle is closed. This notebook does not train, tune, select models, evaluate a split or reopen the spent holdout. The training and evaluation sections execute cells that read the committed record of the runs that actually happened, labelled `RECORDED_TRAINING_EVIDENCE` / `NO_NEW_TRAINING_EXECUTED`; the final-test values are read from published **aggregate reports**, never from holdout data.

**Publication status:** the notebook, its two support modules and the Phase 13B evidence must be published to the repository before the clone cell works on Colab. Local verification does not establish Colab runtime compatibility. The complete MP4 has no public download URL yet; the six committed frames work without it.

In [ ]:
# @title Choose delivery mode (Mode A is the default)
RUN_INFERENCE = False  # @param {type:"boolean"}
MODEL_VIEW = "compare"  # @param ["compare", "detector", "segmenter"]
DEVICE = "cpu"  # @param ["cpu", "cuda"]
print("Mode B enabled" if RUN_INFERENCE else "Mode A: no models will be loaded")

## 1. Visão geral / Overview — clone and load the delivery support code

A fresh Colab runtime already supplies Python and IPython display. **Mode A installs no project packages.** Both support modules use the standard library and import notebook display only when requested. `runpy` loads these standalone modules without importing the scientific package initializer.

`main` is resolved once by cloning; the exact checked-out commit is printed. Existing checkouts are reused without pull/reset. The clone must contain the approved Phase 13B revision and both delivery modules. No data download is performed.

In [ ]:
import os
import runpy
import subprocess
from pathlib import Path

if "CSVISION_ALLOW_TEST_SPLIT" in os.environ:
    raise RuntimeError("CSVISION_ALLOW_TEST_SPLIT must be unset")
REPOSITORY = "https://github.com/Novachrono117/Construction-Safety-Vision-PPE-Detection-Instance-Segmentation-Video-Analytics.git"
ROOT = Path("/content/construction-safety-vision")
EVIDENCE_COMMIT = "fa0bea2b072b124d65f4e9bd1b932a22d98149d2"
if not ROOT.exists():
    subprocess.run(["git", "clone", "--branch", "main", REPOSITORY, str(ROOT)], check=True)
if not (ROOT / ".git").is_dir():
    raise RuntimeError("The target directory is not a Git checkout; use a fresh runtime")
revision = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
print("Repository revision:", revision)
module = ROOT / "src/construction_safety_vision/delivery_demo.py"
academic_module = ROOT / "src/construction_safety_vision/delivery_academic.py"
ready = subprocess.run(
    ["git", "merge-base", "--is-ancestor", EVIDENCE_COMMIT, "HEAD"], cwd=ROOT, capture_output=True
)
if ready.returncode or not module.is_file() or not academic_module.is_file():
    raise RuntimeError(
        "PUBLICATION_REQUIRED: publish Phase 13B and this notebook/support modules, "
        "then use a fresh Colab runtime. No substitute evidence is used."
    )
demo = runpy.run_path(str(module))
academic = runpy.run_path(str(academic_module))
print("Mode A ready. No checkpoint, dataset or inference framework loaded.")

### 1.1 Architecture and published metrics

Final-test and validation values are labelled separately and loaded from their committed JSON records. Box and mask scores remain separate. Source links point to the approved evidence revision.

In [ ]:
demo["show_overview"](ROOT)

## 2. Dataset e classes / Dataset and classes

The modelling population, the five classes and the frozen split sizes, read from the committed population record and the task manifests. Nothing is downloaded and no image is opened; the dataset itself is not redistributed here.

**Source.** Roboflow Universe, workspace `agis-workspace-8gs52`, project `construction-ppe-compliance-detection`, version 4, exported as COCO instance segmentation, licensed **CC BY 4.0**.

**Populations, which are not one number.** The provenance population holds **436 source images**. The **modelling population is 433**: three zero-instance, out-of-domain images were excluded logically after human review, and their files were kept on disk. There are **2031 canonical annotations**. The export also ships offline-augmented copies of training images; they were never counted as independent observations.

**Classes (5).** `person`, `helmet_loose`, `helmet_on_head`, `vest_loose`, `vest_on_body`. The names describe what is visible in the frame. They do not certify compliance, equipment suitability or site safety, and this project holds no compliance ground truth.

**Frozen split.** train **303 images / 1422 annotations**, validation **65 / 304**, test **65 / 305**. Detection and segmentation use exactly the same image ids in each split.

**How the data was prepared, and why it matters for reading the numbers.**

- **Instance segmentation geometry is canonical.** The detection boxes are **derived mathematically from the polygons and masks**, never annotated separately and never taken from the provider's own stored boxes, which disagree with their own geometry inside the export.
- **Confirmed duplicate and near-duplicate images were grouped, and each group was kept inside a single split**, so the same scene cannot straddle a split boundary. This reduces the leakage that was actually detected; it does not certify that two images never share a site, a day, a camera or a worker.
- **The test split was protected** from the moment the split was frozen until one predeclared final evaluation, and is now spent. It took no part in model selection, hyperparameter choice, threshold choice or data cleaning.
- **`vest_loose` is rare**: 8 images in the whole population, split 5 / 1 / 2 over train / validation / test. Every `vest_loose` figure in this notebook is descriptive, carries an explicit small-sample caveat, and decided nothing.

In [ ]:
academic["show_dataset"](ROOT)

## 3. TREINO / TRAINING

`RECORDED_TRAINING_EVIDENCE` · `NO_NEW_TRAINING_EXECUTED`.

**The default notebook does not retrain the models, by design.** Each model was trained exactly once under a protocol frozen before the run, and re-running it would produce a different checkpoint under the same name and invalidate every published result. The cells below execute against the committed artifacts of those runs and print the complete recipe — architecture, image size, epochs, batch, seed, the optimizer that `optimizer: auto` actually resolved to, every augmentation argument, the checkpoint-selection rule, the selected epoch and the frozen model identity — followed by the training and validation curves recorded at the time.

The reproducible entry point and the frozen configuration files are printed with each recipe. Executing them needs a CUDA GPU and the materialised dataset, which this runtime does not have.

### 3.1 Os dois modelos e por que foram escolhidos / The two frozen models and why they were selected

| | **D2 — detector** | **S1 — segmenter** |
| --- | --- | --- |
| Architecture | YOLO11n | YOLO11n-seg |
| Pretrained initialization | `yolo11n.pt` | `yolo11n-seg.pt` |
| Task | object detection (axis-aligned boxes) | instance segmentation |
| Input size | imgsz 768 | imgsz 768 |
| Task-specific arguments | — | `mask_ratio: 4`, `overlap_mask: false` |

**Why D2 was selected.** The detector choice was made on the **validation split only**, under a policy frozen before the candidates ran. The deciding metric was the unweighted mean of per-class AP@0.50:0.95 over the classes meeting a frozen support rule (at least 5 validation images **and** at least 20 validation instances), with a practical margin of 0.005 — an engineering threshold, not a significance test. Three configurations were trained once each: the reference D0 (YOLO11n at imgsz 640) scored 0.570142, the capacity variant D1 (YOLO11s at imgsz 640) scored 0.560017, and D2 — the resolution variant, the same YOLO11n at imgsz 768 — scored 0.594018. D2's **+0.023876** over the reference cleared the margin, and the result was accepted in human review. This is a selection among the configurations studied, with one run per configuration; it is not a claim of a universal optimum, and the holdout took no part in it.

**Why S1 was selected.** The segmenter choice was likewise made on validation only. S1 differs from the segmentation reference S0 in **exactly one declared argument**, `overlap_mask` from `true` to `false`; architecture, resolution, batch, seed, label adapter and checkpoint rule were inherited unchanged and verified. Judged by the same external evaluator against the same canonical masks, the supported-class mean mask AP@0.50:0.95 moved from 0.484643 to **0.559463**, a delta of **+0.074820**, above the same 0.005 margin frozen before the run. The gain is **not uniform and must not be quoted as if it were**: `person` improved by far the most, while `helmet_loose` **regressed**. Why any individual class moved is UNKNOWN.

**What the two models actually output.** This is the representational difference the whole project is about.

- **D2** emits, per object: **class + confidence + a rectangular localization**.
- **S1** emits, per object: **class + confidence + a bounding box + an instance mask**.

A mask additionally supports **foreground area**, **non-rectangular shape**, a **mask centroid** and **overlap / containment analysis** between instances — quantities a rectangle cannot express at all. **This does not make S1 the better model.** The two do not solve the same output task, their recognition performance was broadly similar, and the mask carries a measured cost in latency and inference memory. No winner is declared anywhere in this project.

**Transfer learning.** Neither model was trained from scratch. **D2 is initialized from the pretrained `yolo11n.pt` weights and S1 from the pretrained `yolo11n-seg.pt` weights**, and the PPE dataset is then used to **fine-tune** them: the **train** split supplies the gradient steps and the **validation** split drives the checkpoint rule and every selection decision above. The **final-test split was used for none of it** — not for model selection, not for hyperparameter selection, not for threshold tuning. It was read exactly once, after both models were already frozen.

### 3.2 Hiperparâmetros de treino / Training hyperparameters, as recorded

Both runs completed 100 of 100 epochs; neither stopped early.

| Argument | D2 (detector) | S1 (segmenter) |
| --- | --- | --- |
| epochs | 100 | 100 |
| batch | 16 | 8 |
| imgsz | 768 | 768 |
| seed | 42 | 42 |
| deterministic (requested) | true | true |
| AMP | true | true |
| patience | 50 | 50 |
| optimizer (policy) | auto | auto |
| **resolved optimizer** | **AdamW** | **AdamW** |
| **effective initial learning rate** | **0.001111** | **0.001111** |
| **effective momentum** | **0.9** | **0.9** |
| weight_decay | 0.0005 | 0.0005 |
| warmup_epochs | 3 | 3 |
| lrf (final LR factor) | 0.01 | 0.01 |
| mask_ratio | — | 4 |
| overlap_mask | — | false |
| **selected checkpoint epoch** | **90** | **77** |

**`optimizer: auto` is a policy, not a value, and confusing the two is the easy mistake.** The frozen configuration files declare `lr0: 0.01` and `momentum: 0.937`; those are **inputs to the automatic policy, not a statement of what ran**. The policy resolved to **AdamW at lr0 0.001111 and momentum 0.9** for both models, captured directly from the framework's own log line. Never read the file's `lr0` as the learning rate of either run. The schedule is a linear decay from the effective `lr0` down to `lr0 * lrf`, after a three-epoch warmup.

**Checkpoint selection.** Each run keeps the epoch its framework validation fitness scored highest, computed on the validation split only. For **S1** that fitness is the **unweighted sum of box and mask mAP@0.50:0.95** — a checkpoint-selection mechanism, accepted and recorded before the run, and deliberately **not** the project's reported score. The primary reported segmentation metric remains **mask AP**, so the selected epoch need not be the epoch that maximised it; that gap is disclosed rather than closed after the fact.

The augmentation arguments, including the `close_mosaic: 10` schedule, are listed in 3.3.

### 3.3 Augmentation, as recorded

**Both models inherited the same augmentation set.** These values are read from the frozen training record — the arguments the framework actually resolved and wrote for each run — not from a recommendation or a default table.

| Group | Arguments |
| --- | --- |
| Colour jitter (HSV) | `hsv_h 0.015`, `hsv_s 0.7`, `hsv_v 0.4` |
| Geometry, active | `translate 0.1`, `scale 0.5`, `fliplr 0.5` |
| Geometry, disabled | `degrees 0`, `shear 0`, `perspective 0`, `flipud 0` |
| Composition | `mosaic 1.0`, `close_mosaic 10` (mosaic switched off for the last 10 epochs) |
| Composition, disabled | `mixup 0`, `cutmix 0`, `copy_paste 0` |
| Other | `erasing 0.4`, `auto_augment randaugment` |

Vertical flipping and rotation are off on purpose in the recorded recipe: the value is `0`, not an omission. The two cells below print the complete resolved argument list for each model, together with the recorded execution outcome and the original training and validation curves.

In [ ]:
academic["show_training"](ROOT, "D2")

In [ ]:
academic["show_training"](ROOT, "S1")

## 4. AVALIAÇÃO / EVALUATION

**FINAL TEST RESULTS — READ FROM THE COMMITTED ONE-SHOT EVALUATION.**

The holdout was locked from the split freeze until Phase 11B, read exactly once under a protocol frozen beforehand, and is now spent. The cell below **executes no model and recomputes no metric**: it reads the committed result artifacts by field and renders the canonical COCO average precision, the operating-point precision and recall, the per-class figures, the direct instance-mask IoU diagnostic, the object-level counts, both final-test confusion matrices and the bounded validation-versus-test comparison.

There is no executable final-test path in this notebook and no executable validation path either: reproducing either evaluation needs the materialised dataset and the frozen checkpoints, and the holdout may never be read again.

### 4.1 Como a avaliação é calculada / How the evaluation is computed

**One external evaluator judges both models: `COCOeval` from pycocotools 2.0.11**, run against this project's canonical COCO ground truth. Boxes are scored with `iouType='bbox'`, masks with `iouType='segm'`, at IoU thresholds 0.50 to 0.95 in steps of 0.05. An external evaluator is used precisely because the detector and the segmenter run through different framework validation paths, so their own native numbers are not guaranteed to be computed identically.

- **IoU (intersection over union).** The area shared by two regions divided by the area they cover together. 1.0 is a perfect match, 0.0 no overlap. It is defined for boxes and for masks; the mask version compares pixel sets, which is the stricter question.
- **mAP@0.50.** Average precision — the area under the precision-recall curve — computed with a single IoU threshold of 0.50, so a prediction counts as matched when it overlaps the right object by at least half, then averaged over the classes.
- **mAP@0.50:0.95.** The same average precision computed at ten IoU thresholds from 0.50 to 0.95 and averaged. Each step demands tighter localization, which makes this the stricter measure and the primary one reported here.
- **Precision.** Of the predictions made at the stated operating point, the fraction that matched a canonical object under the stated matching rule.
- **Recall.** Of the canonical objects that exist, the fraction recovered under that same rule.
- **Confusion matrix.** Counts of predicted class against true class, with a background row and column holding predictions that matched no object and objects that received no prediction. **Its matching rule is not the precision/recall rule** — see 4.2.
- **Direct instance-mask IoU diagnostic.** A secondary measurement that is **not** an average precision. Predictions and canonical instances are matched one-to-one inside each image and class by maximising the total IoU; a pair sharing no pixel is not a match. It yields two headline numbers that are **not interchangeable**: `matched_mask_iou_mean` describes mask quality only where the model produced an overlapping same-class instance, while `gt_normalized_mask_iou` divides the same IoU sum by **every** canonical instance, so missed objects count as zero. Precise masks on the objects that were found do not cancel the objects that were missed.

**Boxes and masks are separate tasks and their scores are never merged.** There is no composite, weighted or overall score anywhere in this project, and no winner is declared between D2 and S1.

### 4.2 Pontos de operação / Operating points and matching rules

**Two confidence thresholds exist and they are never mixed.**

| Setting | Value | Where it applies |
| --- | --- | --- |
| Prediction confidence floor for AP | **0.001** | the canonical COCO AP passes only |
| Frozen operational confidence | **0.25** | precision/recall, confusion matrix, object-level counts, the direct mask-IoU diagnostic and video inference |
| IoU for counting precision and recall | **0.50**, class-aware, one-to-one | the operating-point P/R |
| Inference NMS IoU | **0.70** | both models |
| `max_det` | **300** | both models |
| COCOeval `maxDets` | **[1, 10, 100]** | the AP computation |

**Why AP runs at 0.001, and why that is deliberately not an operating point.** Average precision integrates over the whole score curve, so it needs the low-scoring tail; cutting the tail off would truncate the curve rather than clean it up. **0.25** is the project's operating point, frozen before any holdout number existed and applied identically to both models. A figure produced at one threshold is never reported under the other's name, and neither threshold was ever swept or tuned on the test split. Note also that the model proposes up to **300** candidates while COCOeval scores at its conventional cap of **100**; both numbers are recorded rather than reconciled.

**NMS — non-maximum suppression.** A detector proposes many overlapping candidate boxes for the same object. NMS sorts the candidates by confidence, keeps the highest-scoring one, discards every remaining candidate whose IoU with it exceeds the threshold — **0.70** here — and repeats until nothing is left to suppress. `max_det` then caps how many detections survive per image. Without it a single object would be reported several times.

**The confusion matrix does not use the precision/recall rule, and the two sets of counts must not be mixed.** The matrix follows the framework's own detection semantics, read from the installed source rather than assumed: confidence **0.25**, IoU **0.45**, and **class-agnostic** matching with the class pair then recorded — so a matched pair whose classes disagree lands off the diagonal and counts as **both** a false positive and a false negative. Precision and recall instead use IoU **0.50** with **class-aware** one-to-one matching. Same predictions, two different questions, two different answers.

In [ ]:
academic["show_evaluation"](ROOT)

## 5. Evidência qualitativa / Qualitative evidence

First, where the holdout FP/FN examples live and how they were selected. Then a committed **validation** figure: the views preserve the original deterministic selection and do not regenerate predictions or choose new examples. **FP** means an unmatched prediction; **FN** means an unmatched canonical object under the frozen matching rule. No valid example is shown as such.

In [ ]:
academic["show_qualitative"](ROOT)

In [ ]:
GALLERY_VIEW = "D2"  # @param ["D2", "S1", "mask quality", "box versus mask"]
demo["show_gallery"](ROOT, GALLERY_VIEW)

## 6. Evidência em vídeo / Video evidence

Choose one of the three previously reviewed frame pairs. The full output is 104.52 seconds; these figures are the committed evidence, including genuine visible failures. Source footage and derivatives: **Frank Vincentz / CC BY-SA 3.0**. Full attribution is linked in the output.

In [ ]:
FRAME_PAIR = 1  # @param [1, 2, 3] {type:"raw"}
demo["show_video_evidence"](ROOT, FRAME_PAIR)

### Optional: view a copy of the existing final MP4

Full-video publication is pending. If you already have the final MP4, enable this cell and upload it. The recorded size and SHA-256 are checked; no inference occurs. It is approximately 582 MB, so uploading takes time. Browser playback depends on MP4/mp4v support. The committed frame gallery above always remains available. Do not upload research dataset material.


In [ ]:
UPLOAD_RECORDED_VIDEO = False  # @param {type:"boolean"}
if UPLOAD_RECORDED_VIDEO:
    demo["colab_view_final_video"](ROOT)
else:
    print("Optional MP4 upload skipped. Committed figures remain available.")

## 7. INFERÊNCIA / INFERENCE — Mode B optional environment

**Keep RUN_INFERENCE=False for Mode A.** Enabling Mode B installs the repository's existing `uv.lock` environment in an isolated Python 3.12 virtual environment, then runs the frozen detector and segmenter on a video **you** supply. No project dependency or scientific setting is changed. The bootstrap tool is pinned to `uv==0.12.13`; it is the existing repository package manager, not a new model dependency.

The CUDA wheels have a substantial download/disk cost even if CPU is selected. Choose a Colab GPU runtime before selecting `cuda`; an unavailable GPU causes a clear error, never a silent CPU fallback. Colab hardware/runtime compatibility still requires an actual cloud run. The repository's recorded benchmark describes its original local hardware only.

### 7.1 Como a inferência funciona / How the inference pipeline works

**Input:** an MP4 you supply. **Processing:** frame by frame — each decoded frame is passed to the model as a single image.

- **D2** returns, for each detection in the frame: **class, confidence, bounding box**.
- **S1** returns the same three **plus an instance mask**, reconstructed onto the original frame canvas.

The runtime keeps the frozen inference settings and the notebook exposes **no tuning control** over them: **FP32**, **batch 1**, **imgsz 768**, operational confidence **0.25**, **NMS IoU 0.70**, **max_det 300**.

**There is no tracking.** Every frame is processed independently, so no identity carries from one frame to the next: an object that appears, disappears and reappears produces three unrelated detections. Consecutive frames therefore show detection stability, never object permanence.

**No threshold is tuned during the demonstration**, and nothing it produces is a scientific result. It is a user demonstration on footage outside the research dataset, and its measured throughput is not the controlled benchmark.

**Mode B never downloads a checkpoint and has no retraining fallback.** You supply the frozen D2 and/or S1 weights yourself, and the exact recorded byte size and SHA-256 are verified **before the file is deserialized**.

In [ ]:
if RUN_INFERENCE:
    demo["install_runtime"](ROOT)
else:
    print("Mode B installation skipped.")

### 7.2 Upload the frozen checkpoint(s)

Upload D2 for detector, S1 for segmenter, or both when prompted for compare. Only the exact recorded byte sizes and SHA-256 values are accepted **before model loading**. Existing weights are never replaced. No checkpoint download or retraining fallback exists. Checkpoint redistribution is still subject to the repository's licensing policy.


In [ ]:
if RUN_INFERENCE:
    demo["colab_upload_checkpoints"](ROOT, MODEL_VIEW)
else:
    print("Checkpoint upload skipped.")

### 7.3 Upload your external video and execute

Upload an ordinary **MP4, up to 2 GiB**, that you are authorized to use and that is outside the research dataset. Browser upload buffers the file in memory; a short clip is more practical. Constant-rate, even-dimension videos are supported by the existing runtime; unsupported timestamps/codecs fail explicitly.

The runtime preserves its frozen inference settings (no tuning controls), writes an MP4 and provenance JSON, and does not preserve audio. The output is a new user demo, not a new scientific result. Runtime measurements are not the controlled benchmark. Source/license information is marked unspecified in provenance; add attribution before publishing your output.


In [ ]:
CONFIRM_EXTERNAL_VIDEO = False  # @param {type:"boolean"}
result_video = None
if RUN_INFERENCE:
    if not CONFIRM_EXTERNAL_VIDEO:
        raise RuntimeError("Confirm an authorized external video before upload/inference")
    input_video = demo["colab_upload_video"](ROOT)
    result_video = demo["run_external_video"](
        ROOT,
        input_video,
        mode=MODEL_VIEW,
        device=DEVICE,
        confirmed_external=CONFIRM_EXTERNAL_VIDEO,
    )
    print("Complete:", result_video.name)
else:
    print("Video upload and inference skipped.")

### 7.4 Retrieve output and provenance

The runtime writes MP4/mp4v; browser playback is not guaranteed. Download and use a compatible local player if necessary. Download the provenance together with the output. Colab storage is temporary. **Clear all notebook outputs before saving to Git** so uploaded/generated media is not embedded in the repository.


In [ ]:
if RUN_INFERENCE and result_video is not None:
    from google.colab import files

    files.download(str(result_video))
    files.download(str(result_video.with_suffix(".mp4.provenance.json")))
else:
    print("Mode A complete. No inference output was created.")

## 8. Limitações e licenças / Limitations and licences

**Scope.** Every number above is a single run per configuration, evaluated once per split. No significance test was predeclared and none may be added. Why any class or any validation-versus-test gap moved is UNKNOWN. `vest_loose` has 2 holdout images and 7 instances; its figures are descriptive and decide nothing. No compliance ground truth exists in this project, so no safety-compliance accuracy is claimed. Nothing here demonstrates production readiness, real-time throughput or deployment.

**Reproducibility.** Reproducible by design, not yet demonstrated from a clean clone: the training and evaluation entry points, configurations and fingerprints are committed, but the checkpoints and the materialised dataset are not, and the documented command path does not yet cover the whole pipeline.

**Licences.** Dataset: Roboflow Universe `agis-workspace-8gs52/construction-ppe-compliance-detection` v4, CC BY 4.0 — not redistributed by this notebook. Demonstration footage and the derived frames: Frank Vincentz / Wikimedia Commons, CC BY-SA 3.0, overlays added. Checkpoint redistribution remains subject to `delivery/LICENSING.md`; Mode B therefore asks you to supply the frozen weights yourself and verifies their SHA-256 before deserialization. Code is licensed by this repository's `LICENSE`.